# Pair and morph two snapshots

`morph` turns one snapshot into another: each pair of partials becomes a glide
from source to target, and `pair` decides which partials go together. This
notebook morphs `speech1` (41 peaks) into `conga3` (10 peaks) with the
`"equal"`, `"power"`, `"sigmoid"` and `"transport"` pairings, listens to two
morphs and writes one as an events table. Read `visual_tsv` first for
snapshots.

In [ ]:
from pathlib import Path
import numpy as np
import audiospylt as asp

## Source and target

Speech reaches 3055.3 Hz; the conga stays at or below 600.0 Hz.

In [ ]:
speech = asp.read_table("../data/tables/speech1.tsv")
conga = asp.read_table("../data/tables/conga3.tsv")
print(f"speech1 {len(speech)} peaks, {speech.freq.min():.1f}–{speech.freq.max():.1f} Hz")
print(f"conga3 {len(conga)} peaks, {conga.freq.min():.1f}–{conga.freq.max():.1f} Hz")
asp.nb.show(asp.viz.plot_snapshot(speech, conga, labels=["speech1", "conga3"], freq_scale="log"))

## Equal pairing

`"equal"` spreads the 41 speech peaks evenly over the 10 conga peaks, low to
high. `plot_morph_map` draws each pair as a line from source (left) to target
(right); `morph` turns the pairs into events that glide over the given
duration, here 2.0 s. The default `duplicates="full"` (for equal/power/sigmoid)
lets every pairing row carry its partials' full amplitudes; `duplicates="primary"`
zeros the non-primary ends so both sides match the snapshots' levels.
`help(asp.morph)` explains the two rules.

In [ ]:
pairing = asp.pair(speech, conga, method="equal")
print("speech peaks per conga peak:", pairing.groupby("dst").size().tolist())
asp.nb.show(asp.viz.plot_morph_map(pairing))
morphed = asp.morph(speech, conga, 2.0, method="equal")
primary = asp.morph(speech, conga, 2.0, method="equal", duplicates="primary")
print(f"full: {len(morphed)} events, {(morphed.amp_start == 0).sum()} start / "
      f"{(morphed.amp_stop == 0).sum()} end at amp 0")
print(f"primary: {len(primary)} events, {(primary.amp_start == 0).sum()} start / "
      f"{(primary.amp_stop == 0).sum()} end at amp 0")
asp.nb.show(asp.viz.plot_events(morphed))
asp.nb.show(asp.viz.plot_events(primary))
sr_m = 44100
asp.nb.play(asp.synthesize(morphed, sr=sr_m), sr_m, normalize=True)
asp.nb.play(asp.synthesize(primary, sr=sr_m), sr_m, normalize=True)

Both morphs keep 41 events. With `"full"`, no row starts or ends at amplitude 0;
with `"primary"`, 31 rows end at 0 (the non-primary copies onto the 10 conga
peaks). Listen for the quieter arrival of the primary morph.

## Power pairing

`"power"` bends the map with an exponent. `parameter_range` returns the range
in which every conga peak still gets at least one speech peak; the cell draws
the maps at both ends. Without `power=`, `pair` and `morph` use the middle of
the range; one morph below uses the high end.

In [ ]:
lo, hi = asp.parameter_range(speech, conga, "power")
for power in (lo, hi):
    p = asp.pair(speech, conga, method="power", power=power)
    print(f"power={power:.4f}: speech peaks per conga peak {p.groupby('dst').size().tolist()}")
    asp.nb.show(asp.viz.plot_morph_map(p))
m = asp.morph(speech, conga, 2.0, method="power")
m_hi = asp.morph(speech, conga, 2.0, method="power", power=hi)
print(f"default power={m.attrs['morph']['parameter']:.4f}; high-end={m_hi.attrs['morph']['parameter']:.4f}")
asp.nb.show(asp.viz.plot_events(m))
asp.nb.show(asp.viz.plot_events(m_hi))

At power=0.2032, 13 and 15 of the 41 speech peaks go to the two lowest conga
peaks; at 1.8964 the counts grow toward the top, up to 8 on the second-highest
peak. The default morph uses 1.0498; `power=1.8964` uses that high-end map
instead, so more speech peaks glide onto the upper conga partials.

## Sigmoid pairing

`"sigmoid"` bends the map with an S curve; `sigmoid_range` sets how steep it
is. The cell draws both ends of its range, morphs with the default, and once
with the high end of the range.

In [ ]:
lo, hi = asp.parameter_range(speech, conga, "sigmoid")
for sigmoid_range in (lo, hi):
    p = asp.pair(speech, conga, method="sigmoid", sigmoid_range=sigmoid_range)
    print(f"sigmoid_range={sigmoid_range:g}: speech peaks per conga peak {p.groupby('dst').size().tolist()}")
    asp.nb.show(asp.viz.plot_morph_map(p))
m = asp.morph(speech, conga, 2.0, method="sigmoid")
m_hi = asp.morph(speech, conga, 2.0, method="sigmoid", sigmoid_range=hi)
print(f"default sigmoid_range={m.attrs['morph']['parameter']:g}; "
      f"high-end={m_hi.attrs['morph']['parameter']:g}")
asp.nb.show(asp.viz.plot_events(m))
asp.nb.show(asp.viz.plot_events(m_hi))

At 0.01 the map is close to `"equal"` (3 to 5 speech peaks per conga peak); at
5.53 the middle conga peaks take 10 and 11 speech peaks, and the two lowest and
two highest take 1 each. The default morph uses 2.77; `sigmoid_range=5.53`
steepens the S curve so the middle destinations take still more of the map.

## Outside the range

Past the ends of the range some conga peaks get no partner: `pair` warns, and a
morph ends on the peaks that are used. A negative parameter flips the map, so
the lowest speech peaks go to the highest conga peaks. Before you run the cell:
how many conga peaks does `power=50.5` use?

In [ ]:
for method, kw in [("power", {"power": 50.5}), ("sigmoid", {"sigmoid_range": 300}),
                   ("power", {"power": -1.0})]:
    p = asp.pair(speech, conga, method=method, **kw)
    print(f"{kw}: {p.dst.nunique()} of {len(conga)} conga peaks used; "
          f"lowest speech peak → {p.freq_stop.iloc[0]:.1f} Hz")

`power=50.5` uses 3 of the 10 conga peaks and `sigmoid_range=300` uses 4; the
warnings count the unused ones. `power=-1.0` uses all 10, so it does not warn,
but it sends the lowest speech peak, 97.4 Hz, to 600.0 Hz, the top of `conga3`.

## Transport

`"transport"` (new in v1) treats the amplitudes as mass and moves the speech
spectrum onto the conga spectrum, low to high, with the least movement in
frequency. A speech peak can split between two conga peaks, so `pair` can
return more rows than either snapshot has. Before you run the cell: how many
rows does it return here?

In [ ]:
pt = asp.pair(speech, conga, method="transport")
mt = asp.morph(speech, conga, 2.0, method="transport")
print(f"pair {len(pt)} rows, morph {len(mt)} events")
asp.nb.show(asp.viz.plot_morph_map(pt))
asp.nb.show(asp.viz.plot_events_3d(mt))

50 pair rows, 41 events. By default a transport morph keeps a row only where it
carries a partial's amplitude at the start or at the end
(`duplicates="primary"`), so both ends sound at the snapshots' levels.
`help(asp.pair)` lists `weight`, the other transport option.

## Listen to two morphs

Listen for the speech peaks gliding onto the conga partials over 2.0 s. The
renders peak near 0.06, far below full scale, so `nb.play` normalizes them.

In [ ]:
sr = 44100
for label, events in (("equal", morphed), ("transport", mt)):
    audio = asp.synthesize(events, sr=sr)
    print(f"{label}: peak {np.max(np.abs(audio)):.4f}")
    asp.nb.play(audio, sr, normalize=True)

## Write one morph

`write_table` saves the equal morph as a v1 events table under `tsv/`.

In [ ]:
Path("tsv").mkdir(exist_ok=True)
path = asp.write_table(morphed, "tsv/speech_to_conga_equal.tsv")
print(path, "|", path.read_text().splitlines()[0])